# Gold Layer — dim_restaurant (SCD 2) + fact_video_restaurant_mention (design 1.3 / 1.4)

Source: `prod.silver.stg_restaurant_mentions` (LLM extraction). Run **after** `Analytic/gold/dims` (needs `dim_video`).

**Dedup → `restaurant_id` (durable):** nếu Google Places tìm thấy quán (`stg_restaurant_geocode`, `match_status = matched`) thì `restaurant_id = 'place:' || place_id` (cùng place = cùng quán, kể cả khi tên trong video khác nhau). Nếu không: cùng `restaurant_name_normalized` **và** cùng quận (`district` normalize: bỏ dấu, `Q.3` / `quận 3` → `quan 3`) → cùng quán; mention thiếu quận dùng quận duy nhất của tên đó nếu tên chỉ xuất hiện ở 1 quận.

**Address / location:** matched restaurants take Google's name, formatted address, ward, city, latitude / longitude and place_id; the district stays the one stated in the video (Google returns post-2025 wards and no district).

**SCD 2:** address / ward / district / city / latitude / longitude / place_id are tracked (`tracked_checksum`). A change closes the current version (`valid_to = run time`, `is_current = false`) and inserts a new one; a restaurant that is no longer in the source is closed without a new version; `restaurant_key = xxhash64(restaurant_id, valid_from)`. `restaurant_name` is **type 1** (overwrites every version).

**Google Place Details (type 1):** `cuisine`, `cuisine_detail`, `price_band`, `price_min_vnd` / `price_max_vnd`, `google_rating`, `google_rating_count`, `google_maps_uri`, `photo_name` come from `stg_restaurant_place_details` (refreshed monthly) and overwrite every version, so a new rating never opens a new version.
- `cuisine` = the most specific Google type found in `CUISINE_TYPES` (list order), a dish keyword in the unaccented restaurant name (`CUISINE_NAME_KEYWORDS`, e.g. `buffet` → Buffet, `pho`, `bun`, `com tam` → Món Việt) wins over a generic Google type (`GENERIC_CUISINES`: none, `asian_restaurant`); else `Chưa rõ`. `cuisine_detail` = Google's own label (e.g. "Nhà hàng Quảng Đông").
- `price_band` from Google's per-person `priceRange` (midpoint of start / end, or start when open-ended), else from `priceLevel`: `1 · Dưới 100k`, `2 · 100k–300k`, `3 · 300k–600k`, `4 · Trên 600k`, else `Chưa rõ`.

**Google-only restaurants (added 2026-10-31):** places found by Google Maps Text Search (search job / auto_search, `silver.stg_places`) whose `place:<place_id>` no video mentions are added with `has_youtube_review = false` (type 1): name, address, ward, city, coordinates from Google; district = district of the grid keyword that found them (approximate). Restaurants without Place Details take rating / price / types from `stg_places` (same Enterprise fields), so Place Details is not called for them.

**OpenStreetMap (type 1, added 2026-10-31):** each restaurant with coordinates is matched to the nearest place in `silver.stg_osm_restaurants` within **100 m** whose normalized name is equal, contains the other (≥ 4 letters), or is within a small edit distance (≤ max(2, 25 % of the length)). The match gives `osm_key` and `opening_hours`, and fills `cuisine` only after Google and the name rule: Google specific type → name keyword → OSM `cuisine` → Google generic → `Chưa rõ`. `cuisine_source` = `google` / `name` / `osm` (NULL when unknown). Data © OpenStreetMap contributors (ODbL).

**fact_google_review (added 2026-10-31):** transaction fact, grain = one Google review (`review_id`) from `silver.stg_restaurant_place_reviews` (≤ 5 per Place Details read, accumulated). `restaurant_key` = current version of the restaurant with that `place_id` (−1 if none), `publish_date_key` → `dim_date`; measures `rating` (1–5, additive count / average), plus the review text for the dashboard. Full rebuild each run (small table).

**fact_video_restaurant_mention:** factless bridge, grain video × restaurant. `restaurant_key` = the version current when the mention was first found (kept on later runs). `mention_weight = 1 / restaurants in the video` (sums to 1 per video). Loaded with one full-sync MERGE (atomic).


In [ ]:
%run ./_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

DIM = f"{GOLD}.dim_restaurant"
FACT = f"{GOLD}.fact_video_restaurant_mention"
GEOCODE = f"{SILVER}.stg_restaurant_geocode"  # Google Places lookups (stg/restaurant_geocode)
DETAILS = f"{SILVER}.stg_restaurant_place_details"  # Google Place Details (stg/restaurant_place_details)
PLACES = f"{SILVER}.stg_places"  # Google Maps Text Search results (stg/places), Google-only restaurants
OSM = f"{SILVER}.stg_osm_restaurants"  # OpenStreetMap food places (stg/osm_restaurants), fills cuisine / opening hours
REVIEWS = f"{SILVER}.stg_restaurant_place_reviews"  # Google reviews (stg/restaurant_place_details)
REVIEW_FACT = f"{GOLD}.fact_google_review"
OSM_MAX_DISTANCE_M = 100
for src_table, nb_name in [(GEOCODE, "stg/restaurant_geocode"), (DETAILS, "stg/restaurant_place_details")]:
    if not spark.catalog.tableExists(src_table):
        raise RuntimeError(f"{src_table} does not exist: run {nb_name} first")
if not spark.catalog.tableExists(PLACES):  # before the first Places search: no Google-only restaurant
    PLACES = """(SELECT CAST(NULL AS STRING) AS place_id, CAST(NULL AS STRING) AS place_name,
        CAST(NULL AS STRING) AS restaurant_name_normalized, CAST(NULL AS STRING) AS formatted_address,
        CAST(NULL AS STRING) AS ward, CAST(NULL AS STRING) AS district, CAST(NULL AS STRING) AS city,
        CAST(NULL AS DOUBLE) AS latitude, CAST(NULL AS DOUBLE) AS longitude, CAST(NULL AS STRING) AS primary_type,
        CAST(NULL AS ARRAY<STRING>) AS place_types, CAST(NULL AS DOUBLE) AS rating, CAST(NULL AS INT) AS rating_count,
        CAST(NULL AS STRING) AS price_level, CAST(NULL AS BIGINT) AS price_start_vnd, CAST(NULL AS BIGINT) AS price_end_vnd,
        CAST(NULL AS STRING) AS google_maps_uri WHERE false)"""
    print("silver.stg_places does not exist yet: no Google-only restaurants")
if not spark.catalog.tableExists(OSM):  # before the first osm_ingest: no OSM match
    OSM = """(SELECT CAST(NULL AS STRING) AS osm_key, CAST(NULL AS STRING) AS restaurant_name_normalized,
        CAST(NULL AS STRING) AS cuisine, CAST(NULL AS STRING) AS opening_hours, CAST(NULL AS DOUBLE) AS latitude,
        CAST(NULL AS DOUBLE) AS longitude WHERE false)"""
    print("silver.stg_osm_restaurants does not exist yet: no OpenStreetMap match")

# Google place type -> cuisine shown in the dashboard. Order = priority when a place has several types.
CUISINE_TYPES = [
    ("Món Việt", ["vietnamese_restaurant"]),
    ("Món Hoa", ["chinese_restaurant", "cantonese_restaurant", "dim_sum_restaurant", "chinese_noodle_restaurant",
                 "taiwanese_restaurant"]),
    ("Món Nhật", ["japanese_restaurant", "sushi_restaurant", "ramen_restaurant", "japanese_curry_restaurant",
                  "yakiniku_restaurant", "izakaya_restaurant", "tonkatsu_restaurant"]),
    ("Món Hàn", ["korean_restaurant", "korean_barbecue_restaurant"]),
    ("Món Thái", ["thai_restaurant"]),
    ("Món Ấn", ["indian_restaurant", "north_indian_restaurant", "south_indian_restaurant"]),
    ("Món Âu / Mỹ", ["french_restaurant", "italian_restaurant", "pizza_restaurant", "spanish_restaurant",
                     "greek_restaurant", "mediterranean_restaurant", "german_restaurant", "british_restaurant",
                     "european_restaurant", "american_restaurant", "steak_house", "hamburger_restaurant",
                     "mexican_restaurant", "brazilian_restaurant"]),
    ("Món quốc tế khác", ["indonesian_restaurant", "malaysian_restaurant", "filipino_restaurant", "middle_eastern_restaurant",
                          "lebanese_restaurant", "turkish_restaurant", "afghani_restaurant", "african_restaurant"]),
    ("Hải sản", ["seafood_restaurant"]),
    ("Lẩu & nướng", ["hot_pot_restaurant", "barbecue_restaurant"]),
    ("Chay", ["vegan_restaurant", "vegetarian_restaurant"]),
    ("Buffet", ["buffet_restaurant"]),
    ("Ăn nhanh", ["fast_food_restaurant", "sandwich_shop", "snack_bar"]),
    ("Cafe & tráng miệng", ["cafe", "coffee_shop", "tea_house", "dessert_shop", "dessert_restaurant", "bakery",
                            "cake_shop", "ice_cream_shop", "juice_shop", "confectionery"]),
    ("Món Á khác", ["asian_restaurant"]),   # generic: only when no specific type matched
]
# Used when Google has no specific type (most Vietnamese street food is just "restaurant") or only a
# generic one (GENERIC_CUISINES): keywords in the unaccented restaurant name, first match wins
GENERIC_CUISINES = ("Chưa rõ", "Món Á khác")
CUISINE_NAME_KEYWORDS = [
    ("Buffet", r"\\bbuffet\\b"),
    ("Lẩu & nướng", r"\\b(lau|nuong|bbq)\\b"),
    ("Hải sản", r"\\b(hai san|oc)\\b"),
    ("Món Việt", r"\\b(pho|bun|com tam|com|banh mi|banh xeo|banh cuon|banh canh|hu tieu|mi quang|bo kho|chao|xoi|bo ne)\\b"),
    ("Cafe & tráng miệng", r"\\b(cafe|ca phe|coffee|tra sua|che|kem)\\b"),
]
spark.createDataFrame(
    [(t, cuisine, prio) for prio, (cuisine, types) in enumerate(CUISINE_TYPES) for t in types],
    "place_type STRING, cuisine STRING, priority INT",
).createOrReplaceTempView("cuisine_types")

run_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]
print(f"Silver source: {SILVER}.stg_restaurant_mentions")
print(f"Gold target: {DIM}, {FACT}")
print(f"🕒 run_ts (valid_from of new versions): {run_ts}")


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM} (
        restaurant_key             BIGINT,
        restaurant_id              STRING,
        restaurant_name            STRING,
        restaurant_name_normalized STRING,
        address                    STRING,
        ward                       STRING,
        district                   STRING,
        city                       STRING,
        latitude                   DOUBLE,
        longitude                  DOUBLE,
        place_id                   STRING,
        cuisine                    STRING,
        cuisine_detail             STRING,
        price_band                 STRING,
        price_min_vnd              BIGINT,
        price_max_vnd              BIGINT,
        google_rating              DOUBLE,
        google_rating_count        INT,
        google_maps_uri            STRING,
        photo_name                 STRING,
        has_youtube_review         BOOLEAN,
        cuisine_source             STRING,
        opening_hours              STRING,
        osm_key                    STRING,
        valid_from                 TIMESTAMP,
        valid_to                   TIMESTAMP,
        is_current                 BOOLEAN,
        tracked_checksum           STRING,
        etl_date                   TIMESTAMP
    )
    USING DELTA
    COMMENT 'Restaurant dimension (SCD 2 on address / location; name and Google Place Details type 1). One row per version, -1 = Unknown.'
""")

# Tables created before the Place Details columns: add them (type 1, filled by the MERGE below)
TYPE1_DETAIL_COLS = {"cuisine": "STRING", "cuisine_detail": "STRING", "price_band": "STRING",
                     "price_min_vnd": "BIGINT", "price_max_vnd": "BIGINT", "google_rating": "DOUBLE",
                     "google_rating_count": "INT", "google_maps_uri": "STRING", "photo_name": "STRING",
                     "has_youtube_review": "BOOLEAN", "cuisine_source": "STRING", "opening_hours": "STRING",
                     "osm_key": "STRING"}
missing = [c for c in TYPE1_DETAIL_COLS if c not in spark.table(DIM).columns]
if missing:
    spark.sql(f"ALTER TABLE {DIM} ADD COLUMNS ({', '.join(f'{c} {TYPE1_DETAIL_COLS[c]}' for c in missing)})")
    print(f"➕ {DIM}: added columns {missing}")

# --- Every mention with its durable restaurant_id (dedup rule above) ---
spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW mentions_keyed AS
    WITH m AS (
        SELECT *,
            nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                lower(unaccent_vi(coalesce(district, ''))),
                '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                '^(tp\\.?|thanh pho)\\s+', ''),
                '\\s+', ' ')), '') AS district_norm
        FROM {SILVER}.stg_restaurant_mentions
    ),
    single_district AS (
        SELECT restaurant_name_normalized, max(district_norm) AS only_district
        FROM m WHERE district_norm IS NOT NULL
        GROUP BY restaurant_name_normalized HAVING COUNT(DISTINCT district_norm) = 1
    )
    SELECT m.*,
           g.place_id, g.place_name, g.formatted_address, g.ward AS geo_ward, g.city AS geo_city,
           g.latitude, g.longitude,
           -- same Google place = same restaurant; otherwise normalized name + district
           COALESCE('place:' || g.place_id,
                    md5(concat(m.restaurant_name_normalized, '|', coalesce(m.district_norm, s.only_district, '')))) AS restaurant_id
    FROM m
    LEFT JOIN single_district s ON s.restaurant_name_normalized = m.restaurant_name_normalized
    LEFT JOIN {GEOCODE} g
      ON g.restaurant_name_normalized = m.restaurant_name_normalized AND g.district_norm <=> m.district_norm
     AND g.match_status = 'matched'
""")

# --- One row per restaurant_id: latest mention wins (prefer rows with an address) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY restaurant_id ORDER BY place_id IS NULL, address IS NULL, extracted_at DESC, video_id) AS rn
        FROM mentions_keyed
    )
    SELECT restaurant_id,
           COALESCE(place_name, restaurant_name)    AS restaurant_name,
           restaurant_name_normalized,
           COALESCE(formatted_address, address)     AS address,
           COALESCE(geo_ward, ward)                 AS ward,
           district,
           COALESCE(geo_city, city)                 AS city,
           latitude, longitude, place_id,
           md5(concat_ws('||', COALESCE(formatted_address, address), COALESCE(geo_ward, ward), district,
                         COALESCE(geo_city, city), CAST(latitude AS STRING), CAST(longitude AS STRING), place_id)) AS tracked_checksum,
           true AS has_youtube_review
    FROM ranked WHERE rn = 1
    UNION ALL
    -- Google-only: places found by Text Search that no video mentions (same 'place:' id as a geocoded mention)
    SELECT 'place:' || p.place_id, p.place_name, p.restaurant_name_normalized, p.formatted_address, p.ward,
           p.district, p.city, p.latitude, p.longitude, p.place_id,
           md5(concat_ws('||', p.formatted_address, p.ward, p.district, p.city, CAST(p.latitude AS STRING),
                         CAST(p.longitude AS STRING), p.place_id)),
           false
    FROM {PLACES} p
    WHERE p.place_id IS NOT NULL
      AND NOT EXISTS (SELECT 1 FROM mentions_keyed m WHERE m.restaurant_id = 'place:' || p.place_id)
""")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW mention_restaurant AS
    SELECT DISTINCT video_id, restaurant_id, mention_source FROM mentions_keyed
""")
# --- Type 1 attributes from Google Place Details (one row per restaurant_id) ---
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_details AS
    WITH d AS (   -- Place Details when fetched, else the same Enterprise fields from the Text Search result
        SELECT s.restaurant_id, p.primary_type, p.primary_type_display_name, p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, p.photo_name
        FROM restaurant_src s
        JOIN {DETAILS} p ON p.place_id = s.place_id AND p.fetch_status = 'ok'
        UNION ALL
        SELECT s.restaurant_id, p.primary_type, CAST(NULL AS STRING), p.place_types, p.rating, p.rating_count,
               p.price_level, p.price_start_vnd, p.price_end_vnd, p.google_maps_uri, CAST(NULL AS STRING)
        FROM restaurant_src s
        JOIN {PLACES} p ON p.place_id = s.place_id
        LEFT ANTI JOIN {DETAILS} x ON x.place_id = s.place_id AND x.fetch_status = 'ok'
    ),
    typed AS (   -- primary type first, then the other Google types in their order
        SELECT d.restaurant_id, t.place_type, -1 AS pos FROM d LATERAL VIEW explode(array(d.primary_type)) t AS place_type
        UNION ALL
        SELECT d.restaurant_id, t.place_type, t.pos FROM d LATERAL VIEW posexplode(d.place_types) t AS pos, place_type
    ),
    cuisine AS (   -- most specific cuisine first (CUISINE_TYPES order), then Google's type order
        SELECT restaurant_id, min_by(c.cuisine, struct(c.priority, t.pos)) AS cuisine
        FROM typed t JOIN cuisine_types c ON c.place_type = t.place_type
        GROUP BY restaurant_id
    ),
    priced AS (
        SELECT d.*, COALESCE((price_start_vnd + price_end_vnd) / 2, price_start_vnd) AS price_mid_vnd FROM d
    )
    SELECT p.restaurant_id,
           COALESCE(c.cuisine, 'Chưa rõ')                       AS google_cuisine,
           p.primary_type_display_name                          AS cuisine_detail,
           CASE
               WHEN p.price_mid_vnd <  100000 THEN '1 · Dưới 100k'
               WHEN p.price_mid_vnd <  300000 THEN '2 · 100k–300k'
               WHEN p.price_mid_vnd <  600000 THEN '3 · 300k–600k'
               WHEN p.price_mid_vnd >= 600000 THEN '4 · Trên 600k'
               WHEN p.price_level IN ('PRICE_LEVEL_FREE', 'PRICE_LEVEL_INEXPENSIVE') THEN '1 · Dưới 100k'
               WHEN p.price_level = 'PRICE_LEVEL_MODERATE'       THEN '2 · 100k–300k'
               WHEN p.price_level = 'PRICE_LEVEL_EXPENSIVE'      THEN '3 · 300k–600k'
               WHEN p.price_level = 'PRICE_LEVEL_VERY_EXPENSIVE' THEN '4 · Trên 600k'
               ELSE 'Chưa rõ'
           END                                                  AS price_band,
           p.price_start_vnd                                    AS price_min_vnd,
           p.price_end_vnd                                      AS price_max_vnd,
           p.rating                                             AS google_rating,
           p.rating_count                                       AS google_rating_count,
           p.google_maps_uri,
           p.photo_name
    FROM priced p
    LEFT JOIN cuisine c ON c.restaurant_id = p.restaurant_id
""")
# --- OpenStreetMap match: nearest OSM place within OSM_MAX_DISTANCE_M with a similar name ---
deg = OSM_MAX_DISTANCE_M / 111000 * 1.5   # bounding box (degrees) before the exact distance
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_osm AS
    WITH c AS (
        SELECT s.restaurant_id, o.osm_key, o.cuisine AS osm_cuisine, o.opening_hours,
               s.restaurant_name_normalized AS a, o.restaurant_name_normalized AS b,
               2 * 6371000 * asin(sqrt(
                   pow(sin(radians(o.latitude - s.latitude) / 2), 2)
                   + cos(radians(s.latitude)) * cos(radians(o.latitude)) * pow(sin(radians(o.longitude - s.longitude) / 2), 2)
               )) AS distance_m
        FROM restaurant_src s
        JOIN {OSM} o
          ON o.latitude BETWEEN s.latitude - {deg} AND s.latitude + {deg}
         AND o.longitude BETWEEN s.longitude - {deg} AND s.longitude + {deg}
        WHERE s.latitude IS NOT NULL AND s.longitude IS NOT NULL
          AND COALESCE(s.restaurant_name_normalized, '') <> '' AND COALESCE(o.restaurant_name_normalized, '') <> ''
    ),
    ok AS (
        SELECT * FROM c
        WHERE distance_m <= {OSM_MAX_DISTANCE_M}
          AND (a = b
               OR (length(b) >= 4 AND instr(a, b) > 0)
               OR (length(a) >= 4 AND instr(b, a) > 0)
               OR levenshtein(a, b) <= greatest(2, CAST(0.25 * greatest(length(a), length(b)) AS INT)))
    )
    SELECT restaurant_id,
           min_by(osm_key, struct(distance_m, osm_key))       AS osm_key,
           min_by(osm_cuisine, struct(distance_m, osm_key))   AS osm_cuisine,
           min_by(opening_hours, struct(distance_m, osm_key)) AS opening_hours
    FROM ok GROUP BY restaurant_id
""")

# --- Final cuisine: Google specific type > keyword in the name > OSM cuisine > Google generic > Chưa rõ ---
name_rule = " ".join(f"WHEN lower(unaccent_vi(s.restaurant_name)) RLIKE '{rx}' THEN '{cuisine}'"
                     for cuisine, rx in CUISINE_NAME_KEYWORDS)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW restaurant_attributes AS
    WITH a AS (
        SELECT s.restaurant_id, s.restaurant_name, s.has_youtube_review,
               d.google_cuisine, d.cuisine_detail, d.price_band, d.price_min_vnd, d.price_max_vnd, d.google_rating,
               d.google_rating_count, d.google_maps_uri, d.photo_name,
               CASE {name_rule} END AS name_cuisine,
               o.osm_cuisine, o.opening_hours, o.osm_key
        FROM restaurant_src s
        LEFT JOIN restaurant_details d ON d.restaurant_id = s.restaurant_id
        LEFT JOIN restaurant_osm o ON o.restaurant_id = s.restaurant_id
    )
    SELECT restaurant_id, restaurant_name,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN google_cuisine
                WHEN name_cuisine IS NOT NULL THEN name_cuisine
                WHEN osm_cuisine IS NOT NULL THEN osm_cuisine
                ELSE COALESCE(google_cuisine, 'Chưa rõ') END AS cuisine,
           CASE WHEN COALESCE(google_cuisine, 'Chưa rõ') NOT IN {GENERIC_CUISINES} THEN 'google'
                WHEN name_cuisine IS NOT NULL THEN 'name'
                WHEN osm_cuisine IS NOT NULL THEN 'osm'
                WHEN COALESCE(google_cuisine, 'Chưa rõ') <> 'Chưa rõ' THEN 'google' END AS cuisine_source,
           cuisine_detail, COALESCE(price_band, 'Chưa rõ') AS price_band, price_min_vnd, price_max_vnd,
           google_rating, google_rating_count, google_maps_uri, photo_name, has_youtube_review,
           opening_hours, osm_key
    FROM a
""")
print(f"OpenStreetMap matches: {spark.table('restaurant_osm').count()}")
print(f"Restaurants in source: {spark.table('restaurant_src').count()} "
      f"(Google-only: {spark.table('restaurant_src').where('NOT has_youtube_review').count()}) · "
      f"with Google details: {spark.table('restaurant_details').count()}")
display(spark.sql("SELECT cuisine, COUNT(*) AS n FROM restaurant_attributes GROUP BY 1 ORDER BY 2 DESC"))


In [ ]:
# --- SCD 2 MERGE: close the changed current version and insert its new version in one atomic MERGE ---
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING (
        SELECT s.restaurant_id AS merge_key, s.* FROM restaurant_src s
        UNION ALL
        -- changed restaurants appear a second time with a NULL key, so they also hit WHEN NOT MATCHED (new version)
        SELECT NULL AS merge_key, s.*
        FROM restaurant_src s
        JOIN {DIM} d ON d.restaurant_id = s.restaurant_id AND d.is_current AND d.tracked_checksum <> s.tracked_checksum
    ) AS s
    ON t.restaurant_id = s.merge_key AND t.is_current
    WHEN MATCHED AND t.tracked_checksum <> s.tracked_checksum THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
        restaurant_key, restaurant_id, restaurant_name, restaurant_name_normalized, address, ward, district, city,
        latitude, longitude, place_id, valid_from, valid_to, is_current, tracked_checksum, etl_date
    ) VALUES (
        xxhash64(s.restaurant_id, TIMESTAMP'{run_ts}'), s.restaurant_id, s.restaurant_name, s.restaurant_name_normalized,
        s.address, s.ward, s.district, s.city, s.latitude, s.longitude, s.place_id,
        TIMESTAMP'{run_ts}', TIMESTAMP'9999-12-31 00:00:00', true, s.tracked_checksum, current_timestamp()
    )
    -- a restaurant that left the source (e.g. its mentions now resolve to a Google place_id): close it
    WHEN NOT MATCHED BY SOURCE AND t.is_current AND t.restaurant_key <> -1 THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
""")

# --- Type 1: the display name and the Google Place Details overwrite every version ---
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING restaurant_attributes AS s ON t.restaurant_id = s.restaurant_id
    WHEN MATCHED AND NOT (t.restaurant_name <=> s.restaurant_name AND t.cuisine <=> s.cuisine
                          AND t.cuisine_detail <=> s.cuisine_detail AND t.price_band <=> s.price_band
                          AND t.price_min_vnd <=> s.price_min_vnd AND t.price_max_vnd <=> s.price_max_vnd
                          AND t.google_rating <=> s.google_rating AND t.google_rating_count <=> s.google_rating_count
                          AND t.google_maps_uri <=> s.google_maps_uri AND t.photo_name <=> s.photo_name
                          AND t.has_youtube_review <=> s.has_youtube_review
                          AND t.cuisine_source <=> s.cuisine_source AND t.opening_hours <=> s.opening_hours
                          AND t.osm_key <=> s.osm_key) THEN
        UPDATE SET t.restaurant_name = s.restaurant_name, t.cuisine = s.cuisine, t.cuisine_detail = s.cuisine_detail,
                   t.price_band = s.price_band, t.price_min_vnd = s.price_min_vnd, t.price_max_vnd = s.price_max_vnd,
                   t.google_rating = s.google_rating, t.google_rating_count = s.google_rating_count,
                   t.google_maps_uri = s.google_maps_uri, t.photo_name = s.photo_name,
                   t.has_youtube_review = s.has_youtube_review, t.cuisine_source = s.cuisine_source,
                   t.opening_hours = s.opening_hours, t.osm_key = s.osm_key, t.etl_date = current_timestamp()
""")
spark.sql(f"""
    INSERT INTO {DIM} (restaurant_key, restaurant_name, valid_from, valid_to, is_current, etl_date)
    SELECT -1, 'Unknown', TIMESTAMP'1900-01-01 00:00:00', TIMESTAMP'9999-12-31 00:00:00', true, current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {DIM} WHERE restaurant_key = -1)
""")
v = spark.sql(f"SELECT COUNT(*) AS n, COUNT_IF(is_current) AS cur FROM {DIM} WHERE restaurant_key <> -1").collect()[0]
print(f"✅ {DIM}: {v['n']} versions, {v['cur']} current")
display(spark.sql(f"""
    SELECT cuisine, price_band, COUNT(*) AS restaurants, COUNT(google_rating) AS with_rating
    FROM {DIM} WHERE is_current AND restaurant_key <> -1 GROUP BY 1, 2 ORDER BY 1, 2
"""))
display(spark.sql(f"""
    SELECT cuisine_source, COUNT(*) AS restaurants, COUNT(osm_key) AS osm_matched, COUNT(opening_hours) AS with_opening_hours
    FROM {DIM} WHERE is_current AND restaurant_key <> -1 GROUP BY 1 ORDER BY 2 DESC
"""))


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT} (
        video_key      BIGINT,
        restaurant_key BIGINT,
        mention_source STRING,
        mention_count  TINYINT,
        mention_weight DOUBLE,
        etl_date       TIMESTAMP
    )
    USING DELTA
    COMMENT 'Factless bridge: 1 video x 1 restaurant. mention_weight = 1 / restaurants in the video (sums to 1).'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW fact_mention_src AS
    WITH m AS (
        SELECT video_id, restaurant_id, max(mention_source) AS mention_source
        FROM mention_restaurant GROUP BY video_id, restaurant_id
    ),
    existing AS (   -- keep the version that was current when the mention was first found
        SELECT f.video_key, f.restaurant_key, d.restaurant_id
        FROM {FACT} f JOIN {DIM} d ON d.restaurant_key = f.restaurant_key
    )
    SELECT
        COALESCE(dv.video_key, -1)                              AS video_key,
        COALESCE(e.restaurant_key, cur.restaurant_key, -1)      AS restaurant_key,
        m.mention_source,
        CAST(1 AS TINYINT)                                      AS mention_count,
        1.0D / COUNT(*) OVER (PARTITION BY m.video_id)          AS mention_weight,
        current_timestamp()                                     AS etl_date
    FROM m
    LEFT JOIN {GOLD}.dim_video dv ON dv.video_id = m.video_id
    LEFT JOIN existing e ON e.video_key = dv.video_key AND e.restaurant_id = m.restaurant_id
    LEFT JOIN {DIM} cur ON cur.restaurant_id = m.restaurant_id AND cur.is_current
""")
spark.sql(f"""
    MERGE INTO {FACT} AS t
    USING fact_mention_src AS s ON t.video_key = s.video_key AND t.restaurant_key = s.restaurant_key
    WHEN MATCHED AND NOT (t.mention_weight <=> s.mention_weight AND t.mention_source <=> s.mention_source) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
f = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT video_key) AS videos, COUNT(DISTINCT restaurant_key) AS restaurants FROM {FACT}").collect()[0]
print(f"✅ {FACT}: {f['n']} mentions, {f['videos']} videos, {f['restaurants']} restaurants")
print("\n--- dim_restaurant + fact_video_restaurant_mention complete ---")


In [ ]:
# --- fact_google_review: one row per Google review (full rebuild; empty until reviews are read) ---
if spark.catalog.tableExists(REVIEWS):
    spark.sql(f"""
        CREATE OR REPLACE TABLE {REVIEW_FACT}
        COMMENT 'Transaction fact: one Google review (Place Details, <= 5 per read, accumulated). Grain review_id. rating 1-5.'
        AS
        SELECT
            r.review_id,
            COALESCE(d.restaurant_key, -1)                                          AS restaurant_key,
            COALESCE(dd.date_key, -1)                                               AS publish_date_key,
            CAST(r.rating AS TINYINT)                                               AS rating,
            r.review_rank,
            r.review_text,
            r.review_language,
            r.original_text,
            r.original_language,
            r.author_name,
            r.publish_time,
            r.relative_time,
            r.google_maps_uri,
            r.first_seen_at,
            r.last_seen_at,
            current_timestamp()                                                     AS etl_date
        FROM {REVIEWS} r
        LEFT JOIN {DIM} d ON d.place_id = r.place_id AND d.is_current AND d.restaurant_key <> -1
        LEFT JOIN {GOLD}.dim_date dd
          ON dd.date_key = CAST(date_format(from_utc_timestamp(r.publish_time, 'Asia/Ho_Chi_Minh'), 'yyyyMMdd') AS INT)
    """)
    g = spark.sql(f"""
        SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_key) AS restaurants, ROUND(AVG(rating), 2) AS avg_rating,
               COUNT_IF(restaurant_key = -1) AS no_restaurant
        FROM {REVIEW_FACT}""").collect()[0]
    print(f"✅ {REVIEW_FACT}: {g['n']} reviews of {g['restaurants']} restaurants · avg {g['avg_rating']} · "
          f"no current restaurant: {g['no_restaurant']}")
else:
    print(f"{REVIEWS} does not exist yet (stg/restaurant_place_details has not read reviews): {REVIEW_FACT} not built")
print("\n--- fact_google_review complete ---")